# ChromaDB Ingestion — Load Preprocessed Chunks into a Persistent Vector Store

This notebook loads `preprocessed_chunks.json` (18,893 resume chunks) into a **persistent ChromaDB** collection using:
- **LlamaIndex** `TextNode` + `VectorStoreIndex`
- **HuggingFace** `BAAI/bge-base-en-v1.5` embeddings
- **ChromaDB** `PersistentClient` (data saved to `./chroma_db/`)

## Libraries to Install

Run the cell below **once** to install the required packages. After installation, **restart the kernel** before proceeding.

| Package | Purpose |
|---|---|
| `llama-index` | Core LlamaIndex framework (TextNode, VectorStoreIndex, StorageContext) |
| `llama-index-vector-stores-chroma` | ChromaVectorStore integration for LlamaIndex |
| `llama-index-embeddings-huggingface` | HuggingFaceEmbedding wrapper for LlamaIndex |
| `chromadb` | Persistent vector database |

> **Note:** These packages will also pull in transitive dependencies like `torch`, `transformers`, `sentence-transformers`, etc.

In [2]:
# Run this cell ONCE, then restart the kernel
!pip install llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface chromadb --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
  Using cached llama_index-0.14.15-py3-none-any.whl.metadata (13 kB)
  Using cached llama_index_vector_stores_chroma-0.5.5-py3-none-any.whl.metadata (413 bytes)
  Using cached llama_index_embeddings_huggingface-0.6.1-py3-none-any.whl.metadata (458 bytes)
  Using cached chromadb-1.5.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.2 kB)
  Using cached llama_index_cli-0.5.3-py3-none-any.whl.metadata (1.4 kB)
  Using cached llama_index_core-0.14.15-py3-none-any.whl.metadata (2.6 kB)
  Using cached llama_index_embeddings_openai-0.5.1-py3-none-any.whl.metadata (400 bytes)
  Using cached llama_index_indices_managed_llama_cloud-0.9.4-py3-none-any.whl.metadata (3.7 kB)
  Using cached llama_index_llms_openai-0.6.21-py3-none-any.whl.metadata (3.0 kB)
  Using cached llama_index_readers_file-0.5.6-py3-none-any.whl.metadata (5.7 kB)
  Using cached llama_index_readers_llama_parse-0.5.1-py3-none-any.wh

---
## 1. Imports & Configuration

In [1]:
import json
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ── Configuration ──────────────────────────────────────
CHUNKS_FILE      = "preprocessed_chunks.json"
CHROMA_DB_PATH   = "./chroma_db"
COLLECTION_NAME  = "market_reference"
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

print("All imports successful ✔")

All imports successful ✔


---
## 2. Load Preprocessed Chunks

In [2]:
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks):,} chunks")
print(f"Keys per entry: {list(chunks[0].keys())}")
print(f"\nSample entry:")
print(f"  id:       {chunks[0]['id']}")
print(f"  metadata: {chunks[0]['metadata']}")
print(f"  content:  {chunks[0]['content'][:150]}…")

Loaded 18,893 chunks
Keys per entry: ['id', 'content', 'metadata']

Sample entry:
  id:       16852973_chunk_0
  metadata: {'Category': 'HR'}
  content:  HR ADMINISTRATOR MARKETING ASSOCIATE HR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15 years of experience in Hospitality and Custom…


---
## 3. Initialize ChromaDB Persistent Client & Collection

In [3]:
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
chroma_collection = chroma_client.get_or_create_collection(COLLECTION_NAME)

print(f"ChromaDB path:     {CHROMA_DB_PATH}")
print(f"Collection:        {COLLECTION_NAME}")
print(f"Existing vectors:  {chroma_collection.count():,}")

ChromaDB path:     ./chroma_db
Collection:        market_reference
Existing vectors:  18,893


---
## 4. Convert JSON Entries → LlamaIndex TextNodes

In [4]:
nodes = []
for entry in chunks:
    node = TextNode(
        text=entry["content"],
        id_=entry["id"],
        metadata=entry["metadata"],
    )
    nodes.append(node)

print(f"Created {len(nodes):,} TextNodes")
print(f"\nSample node:")
print(f"  id:       {nodes[0].node_id}")
print(f"  metadata: {nodes[0].metadata}")
print(f"  text:     {nodes[0].text[:150]}…")

Created 18,893 TextNodes

Sample node:
  id:       16852973_chunk_0
  metadata: {'Category': 'HR'}
  text:     HR ADMINISTRATOR MARKETING ASSOCIATE HR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15 years of experience in Hospitality and Custom…


---
## 5. Load Embedding Model

In [5]:
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
print(f"Embedding model loaded: {EMBED_MODEL_NAME} ✔")

Embedding model loaded: BAAI/bge-base-en-v1.5 ✔


---
## 6. Index Nodes into ChromaDB

This step embeds all 18,893 nodes and stores them in the ChromaDB collection.  
**This may take several minutes** depending on your hardware (CPU embedding of ~19k chunks).

Because we use `PersistentClient`, the index is automatically saved to `./chroma_db/` — **no re-indexing needed** on future runs.

In [6]:
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes=nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
)

print(f"\nIndexing complete ✔")
print(f"Vectors stored: {chroma_collection.count():,}")
print(f"Persistent data saved to: {CHROMA_DB_PATH}/")

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## 7. Test Query — Retrieve Top 5 Chunks for "Software Engineer"

This cell re-opens the persisted ChromaDB store (no re-indexing) and retrieves the top 5 most similar chunks.

In [7]:
def query_index(query_text: str, top_k: int = 5):
    """Open persisted ChromaDB and retrieve top-k similar chunks."""

    # Re-open persistent store (no re-indexing)
    client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
    collection = client.get_or_create_collection(COLLECTION_NAME)

    vs = ChromaVectorStore(chroma_collection=collection)
    em = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

    idx = VectorStoreIndex.from_vector_store(
        vector_store=vs,
        embed_model=em,
    )

    retriever = idx.as_retriever(similarity_top_k=top_k)
    results = retriever.retrieve(query_text)

    print(f"Query: '{query_text}'  |  Top {top_k} results\n")
    print("=" * 80)
    for i, r in enumerate(results, 1):
        snippet = r.node.text[:200].replace('\n', ' ')
        print(f"\n[{i}] Score: {r.score:.4f}  |  ID: {r.node.node_id}")
        print(f"    Category: {r.node.metadata.get('Category', 'N/A')}")
        print(f"    Content:  {snippet}…")
    print("\n" + "=" * 80)

In [8]:
query_index("Software Engineer")

Query: 'Software Engineer'  |  Top 5 results


[1] Score: 0.5635  |  ID: 51588273_chunk_0
    Category: ENGINEERING
    Content:  SOFTWARE ENGINEERING MANAGER Summary Multifaceted technical career with 15 years' track record of innovation and success. Accomplished, enthusiastic, and driven Software Engineer with a solid history …

[2] Score: 0.5300  |  ID: 28630325_chunk_2
    Category: ENGINEERING
    Content:  and integration of existing third party software systems. Design, document and execute engineering procedures, including customization delivery, escalation and technical modernization enhancements. Co…

[3] Score: 0.5194  |  ID: 28762662_chunk_0
    Category: ENGINEERING
    Content:  SOFTWARE ENGINEERING MANAGER Professional Profile 20 years of software product development experience in broadcast media, video servers, editing, large scale applications, and 24 7 services, with emph…

[4] Score: 0.5111  |  ID: 51588273_chunk_1
    Category: ENGINEERING
    Content:  Name City , 